In [1]:
# Corpus-free distillation of #1914 Acts II + III (CI-5 / #1943)
#
# We derive the four evidential roles + the salience hierarchy
# from synthetic state. No corpus, no API key, no LLM.

import sys
sys.path.insert(0, "/absolute/path/to/2025-Epita-Intelligence-Symbolique")

from argumentation_analysis.core.shared_state import UnifiedAnalysisState
from argumentation_analysis.reporting.restitution.specialist_roles import (
    ROLE_DECISIF, ROLE_CORROBORANT, ROLE_CONTRADICTOIRE, ROLE_NON_DISCRIMINANT,
    ROLE_ORDER, classify_specialist_roles,
)
from argumentation_analysis.reporting.restitution.conclusion_salience import (
    KIND_STRENGTH, KIND_TENSION, KIND_VULNERABILITY,
    assess_conclusion_salience,
)


def quality_entry(v, n=9):
    return {"overall": v * n, "scores": {f"vertu_{i}": v for i in range(n)}}

print("Setup OK — modules loaded, no corpus, no API key.")


Setup OK — modules loaded, no corpus, no API key.


In [1]:
# Case 1: a single DECISIF role from a PL refutation.
#
# The state carries one settled PL entry: satisfiable=False.
# Per #1914, a formal violation is established → the role is decisif.

state_d = UnifiedAnalysisState("synthetic")
state_d.propositional_analysis_results = [
    {"satisfiable": False, "formula": "p => q"}
]
roles_d = classify_specialist_roles(state_d)
print("Roles derived:")
for r in roles_d:
    print(f"  role={r.role!r}  statement={r.statement}")
    print(f"  cites={r.cites}")
print()
print(f"ROLE_ORDER (descending weight): {ROLE_ORDER}")
print("Decisif appears first because a formal violation CHANGES the judgment.")


Roles derived:
  role='decisif'  statement=L'axe PL a réfuté 1 inférence(s) : la mise à l'épreuve formelle établit qu'au moins une inférence testée ne tient pas.
  cites=('PL', 'solveur Tweety')

ROLE_ORDER (descending weight): ('decisif', 'contradictoire', 'corroborant', 'non_discriminant')
Decisif appears first because a formal violation CHANGES the judgment.


In [1]:
# Case 2: a CORROBORANT role — two independent methods agree on weakness.
#
# arg_1 carries (a) a localized fallacy and (b) a measured quality
# fraction strictly under the weak bar (0.25). arg_2 spans the bar
# at 0.7 so the #1942 non-vacuity gate opens the corroboration.

state_c = UnifiedAnalysisState("synthetic")
arg1 = state_c.add_argument("argument faible de référence")
arg2 = state_c.add_argument("argument solide (ouvre la porte non-vacuité)")
state_c.identified_fallacies = {
    "f1": {"type": "ad_hominem", "target_argument_id": arg1}
}
state_c.argument_quality_scores = {
    arg1: quality_entry(0.25),   # fraction 0.25
    arg2: quality_entry(0.70),   # fraction 0.70 → spans weak
}

roles_c = classify_specialist_roles(state_c)
print("Roles derived:")
for r in roles_c:
    print(f"  role={r.role!r}  statement={r.statement[:80]}")
    print(f"  cites={r.cites}")
print()
print("Reading: sophisme localisé + qualité faible sur arg_1 =")
print("  deux méthodes indépendantes s'accordent → CORROBORANT.")


Roles derived:
  role='corroborant'  statement=Les axes sophisme et qualité corroborent la faiblesse de arg_1 (25% du maximum a
  cites=('arg_1', 'sophisme', 'qualite')

Reading: sophisme localisé + qualité faible sur arg_1 =
  deux méthodes indépendantes s'accordent → CORROBORANT.


In [1]:
# Case 3: a CONTRADICTOIRE role — methods disagree on the same argument.
#
# arg_1 carries a localized fallacy BUT a measured quality fraction
# at-or-above the strong bar (0.85). Per #1914, the role is
# contradictoire — the tension is real, must NOT be silently picked.

state_x = UnifiedAnalysisState("synthetic")
arg1 = state_x.add_argument("argument de référence (tension)")
state_x.identified_fallacies = {
    "f1": {"type": "ad_hominem", "target_argument_id": arg1}
}
state_x.argument_quality_scores = {
    arg1: quality_entry(0.85),   # fraction 0.85 → above strong bar
}

roles_x = classify_specialist_roles(state_x)
print("Roles derived:")
for r in roles_x:
    print(f"  role={r.role!r}  statement={r.statement[:120]}")
    print(f"  cites={r.cites}")
print()
print("Reading: sophisme localisé + qualité solide sur arg_1 =")
print("  les axes se contredisent → CONTRADICTOIRE (tension non résolue).")


Roles derived:
  role='contradictoire'  statement=Tension non résolue sur arg_1 : un sophisme y est localisé mais la qualité mesurée est solide (85% du maximum applicable
  cites=('arg_1', 'sophisme', 'qualite')

Reading: sophisme localisé + qualité solide sur arg_1 =
  les axes se contredisent → CONTRADICTOIRE (tension non résolue).


In [1]:
# Case 4 (THE pedagogical core of this notebook): NON-DISCRIMINANT.
#
# Sub-case 4a: the PL axis ran and verified 3 inferences, all
# satisfiable. Per #1914, the axis is non_discriminant. Citing
# such a result as a strength would be the badge-without-derivation
# the issue condemns.

state_nd = UnifiedAnalysisState("synthetic")
state_nd.propositional_analysis_results = [
    {"satisfiable": True, "formula": "p"},
    {"satisfiable": True, "formula": "q"},
    {"satisfiable": True, "formula": "p | ~p"},
]
roles_nd = classify_specialist_roles(state_nd)
print("Sub-case 4a — PL all-true:")
for r in roles_nd:
    print(f"  role={r.role!r}  statement={r.statement[:120]}")

# Sub-case 4b (the #1942 anti-pendule): quality axis where EVERY
# measured fraction is under the weak bar. The axis is vacuous —
# it discriminates nothing.

state_q = UnifiedAnalysisState("synthetic")
arg1 = state_q.add_argument("argument faible 1")
arg2 = state_q.add_argument("argument faible 2")
state_q.argument_quality_scores = {
    arg1: quality_entry(0.10),   # fraction 0.10
    arg2: quality_entry(0.20),   # fraction 0.20
}
# Even with a localized fallacy on arg1, the corroboration is
# suppressed by the #1942 non-vacuity gate (no fraction spans the bar).
state_q.identified_fallacies = {
    "f1": {"type": "ad_hominem", "target_argument_id": arg1}
}
roles_q = classify_specialist_roles(state_q)
print()
print("Sub-case 4b — quality axis vacuous (every fraction < weak bar):")
for r in roles_q:
    print(f"  role={r.role!r}  statement={r.statement[:140]}")
print()
print("Reading: a 'weakness' verdict that holds for 100% of the population")
print("  carries no information. Per #1942, render the vacuity, do NOT")
print("  cite it as proof of anything.")


Sub-case 4a — PL all-true:
  role='non_discriminant'  statement=L'axe PL a vérifié 3 inférence(s), toutes satisfaisables — le test ne distingue rien ici.

Sub-case 4b — quality axis vacuous (every fraction < weak bar):
  role='non_discriminant'  statement=L'axe qualité ne discrimine pas ce run : 2/2 notes mesurées sous le seuil faible (50% du maximum applicable) — la faiblesse mesurée n'y dist

Reading: a 'weakness' verdict that holds for 100% of the population
  carries no information. Per #1942, render the vacuity, do NOT
  cite it as proof of anything.


In [1]:
# Case 5: salience hierarchy (Acte III, #1914) on a multi-role state.
#
# The conclusion ranks findings by evidential weight:
#   P1 decisif → P2 contradictoire → P3 corroborant
# Non-discriminating roles are DELIBERATELY excluded from the ranking.
# Unchallenged strengths earn a P3 (accompanying) slot.

state_s = UnifiedAnalysisState("synthetic")
a_decisif = state_s.add_argument("argument decisif (déclencheur PL)")
a_tension = state_s.add_argument("argument contradictoire (tension)")
a_corro  = state_s.add_argument("argument corroboré (faiblesse)")
a_solid  = state_s.add_argument("argument solide non contesté")
# Decisif: PL refute
state_s.propositional_analysis_results = [{"satisfiable": False, "formula": "p => q"}]
# Tension: fallacy + strong quality on a_tension / Corroborant: fallacy + weak on a_corro
state_s.identified_fallacies = {
    "f1": {"type": "ad_hominem", "target_argument_id": a_tension},
    "f2": {"type": "straw_man",  "target_argument_id": a_corro},
}
state_s.argument_quality_scores = {
    a_tension: quality_entry(0.90),   # fraction 0.90 → contradictoire
    a_corro:   quality_entry(0.20),   # fraction 0.20 → corroborant
    a_solid:   quality_entry(0.95),   # fraction 0.95 → strength (no fallacy)
}

salience = assess_conclusion_salience(state_s, counters_total=0)
print("Ranked findings (P1 decisive → P3 accompanying):")
for i, item in enumerate(salience.ranked, start=1):
    print(f"  P{i}  weight={item.weight}  kind={item.kind}  cites={item.cites}")
    print(f"       {item.statement}")

print()
print("Zero-shot surplus:")
print(f"  established: {len(salience.surplus.established)} items")
for s in salience.surplus.established:
    print(f"    - {s.statement[:100]}")
print(f"  procedural_only: {len(salience.surplus.procedural_only)} items")
print()
print("Reading: the conclusion ranks by portance, not by execution order.")
print("  Non-discriminating axes are excluded — they cannot move the judgment.")


Ranked findings (P1 decisive → P3 accompanying):
  P1  weight=1  kind=vulnerabilite  cites=('PL', 'solveur Tweety')
       L'axe PL a réfuté 1 inférence(s) : la mise à l'épreuve formelle établit qu'au moins une inférence testée ne tient pas.
  P2  weight=2  kind=tension  cites=('arg_2', 'sophisme', 'qualite')
       Tension non résolue sur arg_2 : un sophisme y est localisé mais la qualité mesurée est solide (90% du maximum applicable) — les axes se contredisent.
  P3  weight=3  kind=vulnerabilite  cites=('arg_3', 'sophisme', 'qualite')
       Les axes sophisme et qualité corroborent la faiblesse de arg_3 (20% du maximum applicable) — deux méthodes indépendantes s'accordent.
  P4  weight=3  kind=force  cites=('arg_4', 'qualite')
       arg_4 tient : qualité mesurée solide (95% du maximum applicable) et aucun axe ne la conteste.

Zero-shot surplus:
  established: 1 items
    - L'axe PL a réfuté 1 inférence(s) : la mise à l'épreuve formelle établit qu'au moins une inférence te
  procedur

In [1]:
# Conclusion: the 4-role derivation + the salience hierarchy.
#
# What the upstream CoursIA notebook teaches today (5 August) is the
# 3-act restitution WITHOUT the role-based reading. A reader sees a
# list of badges — "PL verified 4 inferences" sits next to "localized
# ad hominem on arg_2" sits next to "quality 0.85 on arg_4" — and
# cannot tell which finding CHANGES the judgment.
#
# #1914 fixes this with two deterministic derivations:
#   1. classify_specialist_roles(state) — each result gets one of
#      decisif / contradictoire / corroborant / non_discriminant.
#   2. assess_conclusion_salience(state) — the conclusion ranks by
#      evidential weight and states the zero-shot surplus.
#
# Anti-pendule: neither call asks the LLM. The hierarchy is in the
# render, not in the prompt. The pedagogical core is the role
# non_discriminant — the case where the result ran but moved nothing,
# which the upstream notebook teaches as a strength.
#
# This notebook is corpus-free: no extract from the dataset, no source
# name, no API key. All state is synthetic, all derivations are
# deterministic, all outputs reproduce.
print("Done. The notebook derives the four roles + the salience hierarchy")
print("from synthetic state, without any corpus or API key.")


Done. The notebook derives the four roles + the salience hierarchy
from synthetic state, without any corpus or API key.
